# Stored execution evidence — Ultrasound Nerve Segmentation

This public evidence copy preserves every textual output cell supplied in `agent_final_submission.zip`.
The original notebook SHA-256 is `58a74d317c99923bfa8e3fb2574fc250954ecfd604443fc1ee008f6c4803262d`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `main` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. Add `KAGGLE_API_TOKEN` in Colab Secrets and accept the Plant Pathology competition rules on Kaggle before downloading.
3. Select a GPU runtime for real training.
4. Run the cells in order.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


def normalize_repo_url(value: str) -> str:
    value = (value or "").strip()

    markdown_match = re.fullmatch(
        r"\[(.*?)\]\((https?://[^)]+)\)",
        value,
    )
    if markdown_match:
        return markdown_match.group(2)

    plain_url_match = re.search(r"https?://\S+", value)
    if plain_url_match:
        return plain_url_match.group(0)

    return value


REPO_URL = normalize_repo_url(
    os.getenv(
        "JIAOZI_REPO_URL",
        "https://github.com/Isso-W/Jiaozi.git",
    )
)
REPO_REF = os.getenv("JIAOZI_REPO_REF", "main")
REPO_DIR = Path("/content/Jiaozi")

# Important:
# Colab may still be inside /content/Jiaozi from a previous run.
# Move out before deleting and cloning the repository again.
os.chdir("/content")

if REPO_DIR.exists():
    print("Removing existing repository:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    cwd="/content",
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)

print("=== git stderr ===")
print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        "Git clone failed.\n"
        f"Return code: {completed.returncode}\n"
        f"stderr:\n{completed.stderr}"
    )

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

requirements_path = REPO_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(
        f"requirements.txt was not found at {requirements_path}"
    )

print("\n=== requirements.txt ===")
print(requirements_path.read_text(encoding="utf-8")[:4000])

print("\n=== installing requirements ===")

pip_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements_path),
    ],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)

print("=== pip stdout ===")
print(pip_result.stdout)

print("=== pip stderr ===")
print(pip_result.stderr)

print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError(
        "pip install failed. Check the pip output above."
    )

print("Jiaozi repository:", REPO_DIR)
print("Jiaozi repo URL:", REPO_URL)
print("Jiaozi ref:", REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))

print("pipeline.py candidates:")
for path in pipeline_candidates:
    print(" -", path)

if not pipeline_candidates:
    raise FileNotFoundError(
        "No pipeline.py was found after cloning the repository."
    )


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
=== pip stdout ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 106.4 MB/s eta 0

In [2]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token


## Download the Plant Pathology data

These cells configure the Kaggle CLI, download `plant-pathology-2020-fgvc7`, and extract the competition files under `/content/data`. The pipeline cell later converts the one-hot `train.csv` into a single `label` column that Jiaozi's local CSV loader can consume.


In [3]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# Upgrade the packaging tools first
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# Install the Kaggle CLI without -q so failures remain visible
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# Read Kaggle access token from Colab Secrets
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# Verify that the Kaggle CLI is available
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle
=== pip stdout

CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [4]:
# Download Kaggle Ultrasound Nerve Segmentation data

KAGGLE_COMPETITION = "ultrasound-nerve-segmentation"

!mkdir -p /content/data/ultrasound-nerve

!kaggle competitions download \
    -c {KAGGLE_COMPETITION} \
    -p /content/data/ultrasound-nerve \
    --force


100% 2.11G/2.11G [01:44<00:00, 21.7MB/s]



In [5]:
# Extract Ultrasound Nerve Segmentation competition files

from pathlib import Path
import shutil
import zipfile

DATA_ROOT = Path("/content/data/ultrasound-nerve")

KAGGLE_DATA_DIR = (
    DATA_ROOT / "ultrasound-nerve-segmentation"
)

DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if KAGGLE_DATA_DIR.exists():
    shutil.rmtree(KAGGLE_DATA_DIR)

KAGGLE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

outer_archives = sorted(
    DATA_ROOT.glob("*.zip")
)

if not outer_archives:
    raise FileNotFoundError(
        f"No downloaded ZIP file found in {DATA_ROOT}"
    )

print("Outer archives:")
for path in outer_archives:
    print(" -", path)

for archive_path in outer_archives:
    print(
        "Extracting:",
        archive_path,
        "->",
        KAGGLE_DATA_DIR,
    )

    with zipfile.ZipFile(
        archive_path,
        "r",
    ) as archive:
        archive.extractall(
            KAGGLE_DATA_DIR
        )


# Recursively extract train.zip, test.zip, etc.
processed_archives = set()

while True:
    nested_archives = [
        path
        for path in KAGGLE_DATA_DIR.rglob("*.zip")
        if path.resolve() not in processed_archives
    ]

    if not nested_archives:
        break

    for archive_path in sorted(nested_archives):
        processed_archives.add(
            archive_path.resolve()
        )

        if archive_path.name.lower().endswith(
            ".csv.zip"
        ):
            target_dir = archive_path.parent
        else:
            target_dir = (
                archive_path.parent
                / archive_path.stem
            )

            target_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

        print(
            "Extracting nested archive:",
            archive_path,
            "->",
            target_dir,
        )

        with zipfile.ZipFile(
            archive_path,
            "r",
        ) as archive:
            archive.extractall(target_dir)


IMAGE_EXTENSIONS = {
    ".tif",
    ".tiff",
    ".png",
    ".jpg",
    ".jpeg",
}


def collect_images(root: Path):
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


all_images = collect_images(
    KAGGLE_DATA_DIR
)

mask_images = [
    path
    for path in all_images
    if path.stem.endswith("_mask")
]

non_mask_images = [
    path
    for path in all_images
    if not path.stem.endswith("_mask")
]


def find_named_image_dir(
    root: Path,
    name: str,
):
    candidates = []

    for directory in root.rglob("*"):
        if not directory.is_dir():
            continue

        if directory.name.lower() != name.lower():
            continue

        images = collect_images(directory)

        if images:
            candidates.append(
                (
                    len(images),
                    directory,
                )
            )

    if not candidates:
        return None

    candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    return candidates[0][1]


TRAIN_ROOT = find_named_image_dir(
    KAGGLE_DATA_DIR,
    "train",
)

TEST_DIR = find_named_image_dir(
    KAGGLE_DATA_DIR,
    "test",
)

if TRAIN_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the extracted train directory."
    )

if TEST_DIR is None:
    raise FileNotFoundError(
        "Could not locate the extracted test directory."
    )

train_files = collect_images(TRAIN_ROOT)

train_masks = [
    path
    for path in train_files
    if path.stem.endswith("_mask")
]

train_images = [
    path
    for path in train_files
    if not path.stem.endswith("_mask")
]

test_images = [
    path
    for path in collect_images(TEST_DIR)
    if not path.stem.endswith("_mask")
]

sample_candidates = sorted(
    path
    for path in KAGGLE_DATA_DIR.rglob("*.csv")
    if "sample" in path.name.lower()
    and "submission" in path.name.lower()
)

if not sample_candidates:
    raise FileNotFoundError(
        "Could not locate sample_submission.csv."
    )

SAMPLE_SUBMISSION_PATH = sample_candidates[0]

print("\nKAGGLE_DATA_DIR       :", KAGGLE_DATA_DIR)
print("TRAIN_ROOT            :", TRAIN_ROOT)
print("TEST_DIR              :", TEST_DIR)
print("Training images       :", len(train_images))
print("Training masks        :", len(train_masks))
print("Test images           :", len(test_images))
print("Sample submission     :", SAMPLE_SUBMISSION_PATH)


# Verify image-mask pairs.
mask_by_base = {
    path.stem.removesuffix("_mask"): path
    for path in train_masks
}

missing_masks = [
    path.name
    for path in train_images
    if path.stem not in mask_by_base
]

if missing_masks:
    raise RuntimeError(
        "Some training images have no matching masks.\n"
        f"First missing masks: {missing_masks[:10]}"
    )

print(
    "\n✅ Ultrasound image-mask pairs "
    "were located successfully."
)


Outer archives:
 - /content/data/ultrasound-nerve/ultrasound-nerve-segmentation.zip
Extracting: /content/data/ultrasound-nerve/ultrasound-nerve-segmentation.zip -> /content/data/ultrasound-nerve/ultrasound-nerve-segmentation

KAGGLE_DATA_DIR       : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation
TRAIN_ROOT            : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/train
TEST_DIR              : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/test
Training images       : 5635
Training masks        : 5635
Test images           : 5508
Sample submission     : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/sample_submission.csv

✅ Ultrasound image-mask pairs were located successfully.


## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated pipeline

This cell runs the integrated Jiaozi flow against the local Kaggle files:

1. Convert Plant Pathology's one-hot labels into a local CSV with `image_id,label`.
2. Run Module 1 from the natural-language `QUERY`.
3. Run Module 2 on sampled real images from the Kaggle folder.
4. Run Module 3 retrieval and optional recommender re-ranking.
5. Run Module 4 code generation with local CSV training metadata.

Set `REAL_TRAINING = True` to generate real-training code and train it in the next cell. With `REAL_TRAINING = False`, Module 4 keeps the generated project in smoke-test mode.


In [15]:
# Ultrasound Nerve Segmentation data -> Jiaozi Modules 1-4

import json
import os
import shutil
import sys
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"

if not PIPELINE_PATH.is_file():
    raise FileNotFoundError(
        f"pipeline.py not found: {PIPELINE_PATH}\n"
        "Run the GitHub clone Cell first."
    )

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Current working directory:", Path.cwd())
print("Using pipeline:", PIPELINE_PATH)


# ============================================================
# 1. Task prompt
# ============================================================

QUERY = (
    "Build a binary semantic segmentation pipeline for the Kaggle "
    "Ultrasound Nerve Segmentation dataset. Each grayscale ultrasound "
    "image has a paired binary mask marking the brachial plexus nerve. "
    "The target region is small, low-contrast, and affected by ultrasound "
    "speckle noise, with many empty masks and strong foreground-background "
    "imbalance. Recommend a suitable medical image segmentation model from "
    "the RAG knowledge base, preferably U-Net or another lightweight "
    "encoder-decoder architecture. Use paired image-mask augmentation, "
    "patient-level validation splitting, and Dice-based or BCE-plus-Dice "
    "training loss. The official evaluation metric is Dice Coefficient and "
    "must be maximized. The model must output pixel-level masks rather than "
    "image-level class predictions, and the pipeline should recommend the "
    "training hyperparameters and save the checkpoint with the best "
    "validation Dice."
)

REAL_TRAINING = True
EPOCHS = 30

OUTPUT_DIR = Path(
    "/content/jiaozi_generated_pipeline"
)


# ============================================================
# 2. Build stable flat image/mask directories
# ============================================================

DATASET_DIR = Path(KAGGLE_DATA_DIR)

FLAT_IMAGE_DIR = (
    DATASET_DIR / "jiaozi_train_images"
)

FLAT_MASK_DIR = (
    DATASET_DIR / "jiaozi_train_masks"
)

if FLAT_IMAGE_DIR.exists():
    shutil.rmtree(FLAT_IMAGE_DIR)

if FLAT_MASK_DIR.exists():
    shutil.rmtree(FLAT_MASK_DIR)

FLAT_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FLAT_MASK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

mask_by_base = {
    path.stem.removesuffix("_mask"): path
    for path in train_masks
}

manifest_rows = []

for image_path in train_images:
    image_id = image_path.stem
    mask_path = mask_by_base.get(image_id)

    if mask_path is None:
        raise FileNotFoundError(
            f"Mask missing for {image_path}"
        )

    image_destination = (
        FLAT_IMAGE_DIR / image_path.name
    )

    mask_destination = (
        FLAT_MASK_DIR
        / f"{image_id}_mask{mask_path.suffix.lower()}"
    )

    shutil.copy2(
        image_path,
        image_destination,
    )

    shutil.copy2(
        mask_path,
        mask_destination,
    )

    with Image.open(image_destination) as image:
        image_size = image.size

    with Image.open(mask_destination) as mask:
        mask_array = np.asarray(
            mask.convert("L"),
            dtype=np.uint8,
        )

        mask_size = mask.size

    if image_size != mask_size:
        raise ValueError(
            f"Image-mask size mismatch for {image_id}: "
            f"{image_size} vs {mask_size}"
        )

    patient_id = (
        image_id.split("_")[0]
        if "_" in image_id
        else image_id
    )

    mask_present = int(
        np.any(mask_array > 0)
    )

    foreground_fraction = float(
        np.mean(mask_array > 0)
    )

    manifest_rows.append(
        {
            "image": image_destination.name,
            "mask": mask_destination.name,
            "patient_id": patient_id,
            "mask_present": mask_present,
            "foreground_fraction": foreground_fraction,
            "label": mask_present,
        }
    )


manifest = pd.DataFrame(
    manifest_rows
)

processed_train_csv = (
    DATASET_DIR
    / "jiaozi_ultrasound_segmentation.csv"
)

manifest.to_csv(
    processed_train_csv,
    index=False,
)

TRAIN_IMAGE_DIR = FLAT_IMAGE_DIR
TRAIN_MASK_DIR = FLAT_MASK_DIR

print("Training manifest:", processed_train_csv)
print("Training pairs   :", len(manifest))
print("Patients         :", manifest["patient_id"].nunique())
print(
    "Masks present    :",
    int(manifest["mask_present"].sum()),
)
print(
    "Empty masks      :",
    int((manifest["mask_present"] == 0).sum()),
)
print(
    "Mean foreground fraction:",
    manifest["foreground_fraction"].mean(),
)

display(manifest.head())


# ============================================================
# 3. Local benchmark information
# ============================================================

LOCAL_DATA_INFO = {
    "benchmark": "ultrasound_nerve_segmentation",
    "competition": "ultrasound-nerve-segmentation",
    "task_type": "image_segmentation",
    "problem_type": "binary_semantic_segmentation",
    "train_csv": str(processed_train_csv),
    "image_dir": str(TRAIN_IMAGE_DIR),
    "mask_dir": str(TRAIN_MASK_DIR),
    "test_dir": str(TEST_DIR),
    "sample_submission": str(SAMPLE_SUBMISSION_PATH),
    "image_column": "image",
    "mask_column": "mask",
    "label_column": "label",
    "group_column": "patient_id",
    "image_path_template": "{image}",
    "mask_path_template": "{mask}",
    "image_extension": "",
    "mask_extension": "",
    "input_channels": 1,
    "output_channels": 1,
    "num_classes": 1,
    "metric": "dice",
    "evaluation_metric": "dice",
    "metric_direction": "maximize",
    "paired_masks": True,
}

print(
    "\nPrepared local benchmark:"
)

print(
    json.dumps(
        LOCAL_DATA_INFO,
        indent=2,
        ensure_ascii=False,
    )
)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)


# ============================================================
# 4. Import Jiaozi modules
# ============================================================

# Always use a brand-new ChromaDB directory.
# This avoids reusing an incomplete or incompatible database.
CHROMA_DIR = Path(
    f"/content/jiaozi_chroma_ultrasound_{uuid.uuid4().hex[:8]}"
)

CHROMA_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

print(
    "Fresh ChromaDB directory:",
    CHROMA_DIR,
)

from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import merge_modules, run_module4_generation
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)


# ============================================================
# 5. Module 1
# ============================================================

print(
    "\n[Notebook] Module 1: "
    "Parsing ultrasound segmentation requirements..."
)

m1_output = module1_pipeline(
    QUERY
)

if m1_output is None:
    raise RuntimeError(
        "Module 1 failed."
    )

m1_output["task_type"] = (
    "image_segmentation"
)

m1_output["evaluation_metric"] = (
    "dice"
)

m1_output["metric_direction"] = (
    "maximize"
)

m1_output["num_classes"] = 1


# ============================================================
# 6. Module 2 wrapper
# ============================================================

class LocalUltrasoundSplit:
    column_names = [
        "image",
        "label",
    ]

    def __init__(
        self,
        frame,
        info,
    ):
        self.frame = frame.reset_index(
            drop=True
        )

        self.image_root = Path(
            info["image_dir"]
        )

        self.labels = (
            self.frame[
                info["label_column"]
            ]
            .astype(int)
            .tolist()
        )

        self.image_paths = [
            self.image_root
            / str(filename)
            for filename in self.frame[
                info["image_column"]
            ].tolist()
        ]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, key):
        if key == "label":
            return self.labels

        if key == "image":
            return [
                self._open_image(path)
                for path in self.image_paths
            ]

        index = int(key)

        return {
            "image": self._open_image(
                self.image_paths[index]
            ),
            "label": self.labels[index],
        }

    @staticmethod
    def _open_image(path):
        with Image.open(path) as image:
            return image.convert("RGB")


def build_local_module2_dataset(
    info,
):
    frame = pd.read_csv(
        info["train_csv"]
    )

    required_columns = {
        info["image_column"],
        info["label_column"],
    }

    missing_columns = (
        required_columns
        - set(frame.columns)
    )

    if missing_columns:
        raise ValueError(
            "Manifest is missing columns: "
            f"{sorted(missing_columns)}"
        )

    sample_size = min(
        600,
        len(frame),
    )

    sampled = frame.sample(
        n=sample_size,
        random_state=42,
    )

    print(
        "Module 2 sample:",
        f"{len(sampled)}/{len(frame)}",
    )

    return {
        "train": LocalUltrasoundSplit(
            sampled,
            info,
        )
    }


# ============================================================
# 7. Module 2
# ============================================================

print(
    "\n[Notebook] Module 2: "
    "Analyzing sampled ultrasound images..."
)

m2_report = (
    ImageStatisticsAnalyzer()
    .analyze(
        build_local_module2_dataset(
            LOCAL_DATA_INFO
        )
    )
)

m2_report.update(
    {
        "total_images": int(
            len(manifest)
        ),
        "num_classes": 1,
        "task_type": (
            "image_segmentation"
        ),
        "problem_type": (
            "binary_semantic_segmentation"
        ),
        "paired_masks": True,
        "input_channels": 1,
        "output_channels": 1,
        "num_patients": int(
            manifest[
                "patient_id"
            ].nunique()
        ),
        "positive_masks": int(
            manifest[
                "mask_present"
            ].sum()
        ),
        "empty_masks": int(
            (
                manifest[
                    "mask_present"
                ]
                == 0
            ).sum()
        ),
        "mean_foreground_fraction": float(
            manifest[
                "foreground_fraction"
            ].mean()
        ),
        "class_distribution": {
            "empty_mask": int(
                (
                    manifest[
                        "mask_present"
                    ]
                    == 0
                ).sum()
            ),
            "nerve_present": int(
                manifest[
                    "mask_present"
                ].sum()
            ),
        },
    }
)


# ============================================================
# 8. Merge Modules 1 and 2
# ============================================================

m3_input = merge_modules(
    m1_output,
    m2_report,
)

m3_input.update(
    {
        "task_type": (
            "image_segmentation"
        ),
        "problem_type": (
            "binary_semantic_segmentation"
        ),
        "evaluation_metric": "dice",
        "metric_direction": "maximize",
        "num_classes": 1,
        "paired_masks": True,
    }
)

constraints = m3_input.setdefault(
    "constraints",
    {}
)

constraints.update(
    {
        "medical": True,
        "class_imbalance": True,
        "small_foreground": True,
        "low_contrast": True,
        "paired_masks": True,
    }
)

print(
    "\n[Notebook] Merged:",
    f"task={m3_input.get('task_type')}",
    f"size={m3_input.get('data_size')}",
    f"metric={m3_input.get('evaluation_metric')}",
)


# ============================================================
# 9. Module 3
# ============================================================

print(
    "\n[Notebook] Module 3: "
    "Retrieving segmentation configurations..."
)

graph = build_graph()

try:
    collection = build_vector_index(
        persist_path=str(
            CHROMA_DIR
        )
    )

except Exception as exc:
    print(
        "[Notebook] First ChromaDB initialization failed:",
        repr(exc),
    )

    print(
        "[Notebook] Retrying with another "
        "fresh database directory..."
    )

    CHROMA_DIR = Path(
        "/content/"
        f"jiaozi_chroma_ultrasound_retry_"
        f"{uuid.uuid4().hex[:8]}"
    )

    CHROMA_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )

    collection = build_vector_index(
        persist_path=str(
            CHROMA_DIR
        )
    )


recommendations = retrieve_top3_hybrid(
    m3_input,
    graph,
    collection,
)

if not recommendations:
    raise RuntimeError(
        "Module 3 returned no segmentation "
        "recommendations. Do not continue to Module 4."
    )

print_results(
    m3_input,
    recommendations,
    graph,
)

try:
    recommendations = recommend(
        recommendations,
        m2_report,
        m3_input,
        memory=OutcomeMemory(),
    )

    print(
        "[Notebook] Recommender "
        "re-ranked candidates."
    )

except Exception as exc:
    print(
        "[Notebook] Recommender skipped:",
        exc,
    )


# ============================================================
# 10. Build task lists
# ============================================================

task_lists = build_all_task_lists(
    recommendations,
    graph,
    fmt="nl",
)

if not task_lists:
    raise RuntimeError(
        "No Module 4 task lists were created."
    )

for task_list in task_lists:
    model_config = task_list.get(
        "model_config"
    )

    if not isinstance(
        model_config,
        dict,
    ):
        continue

    model_config.update(
        {
            "task_type": (
                "image_segmentation"
            ),
            "problem_type": (
                "binary_semantic_segmentation"
            ),
            "train_csv": str(
                processed_train_csv
            ),
            "image_dir": str(
                TRAIN_IMAGE_DIR
            ),
            "mask_dir": str(
                TRAIN_MASK_DIR
            ),
            "test_dir": str(
                TEST_DIR
            ),
            "sample_submission": str(
                SAMPLE_SUBMISSION_PATH
            ),
            "image_column": "image",
            "mask_column": "mask",
            "label_column": "label",
            "group_column": "patient_id",
            "image_path_template": (
                "{image}"
            ),
            "mask_path_template": (
                "{mask}"
            ),
            "image_extension": "",
            "mask_extension": "",
            "input_channels": 1,
            "output_channels": 1,
            "num_classes": 1,
            "paired_masks": True,
            "binary_segmentation": True,
            "grayscale": True,
            "evaluation_metric": "dice",
            "metric": "dice",
            "metric_name": "dice",
            "metric_direction": (
                "maximize"
            ),
            "loss": "bce_dice_loss",
            "loss_function": (
                "bce_dice_loss"
            ),
            "validation_split_unit": (
                "patient_id"
            ),
            "prediction_threshold": 0.5,
            "preserve_original_size": True,
            "offline_smoke": False,
            "recommended_epochs": EPOCHS,
            "competition": (
                "ultrasound-nerve-segmentation"
            ),
            "benchmark_key": (
                "ultrasound_nerve_segmentation"
            ),
        }
    )


print(
    "\nTop model configuration:"
)

print(
    json.dumps(
        task_lists[0][
            "model_config"
        ],
        indent=2,
        ensure_ascii=False,
    )[:5000]
)


# ============================================================
# 11. Module 4
# ============================================================

print(
    "\n[Notebook] Module 4: "
    "Generating real segmentation project..."
)

module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=180,
    llm_provider="openai",
)

summary_path = (
    OUTPUT_DIR
    / "module4_summary.json"
)

if not summary_path.is_file():
    summary_path.write_text(
        json.dumps(
            module4["summary"],
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

DATASET = str(
    processed_train_csv
)

print(
    "\nModule 4 summary:",
    summary_path,
)

print(
    "Generated project output dir:",
    OUTPUT_DIR,
)

print(
    "DATASET:",
    DATASET,
)


Current working directory: /content/Jiaozi
Using pipeline: /content/Jiaozi/pipeline.py
Training manifest: /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_ultrasound_segmentation.csv
Training pairs   : 5635
Patients         : 47
Masks present    : 2323
Empty masks      : 3312
Mean foreground fraction: 0.012058904221358708


,image,mask,patient_id,mask_present,foreground_fraction,label
0,10_1.tif,10_1_mask.tif,10,1,0.032549,1
1,10_10.tif,10_10_mask.tif,10,0,0.000000,0
2,10_100.tif,10_100_mask.tif,10,1,0.050041,1
3,10_101.tif,10_101_mask.tif,10,0,0.000000,0
4,10_102.tif,10_102_mask.tif,10,0,0.000000,0



Prepared local benchmark:
{
  "benchmark": "ultrasound_nerve_segmentation",
  "competition": "ultrasound-nerve-segmentation",
  "task_type": "image_segmentation",
  "problem_type": "binary_semantic_segmentation",
  "train_csv": "/content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_ultrasound_segmentation.csv",
  "image_dir": "/content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_images",
  "mask_dir": "/content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_masks",
  "test_dir": "/content/data/ultrasound-nerve/ultrasound-nerve-segmentation/test",
  "sample_submission": "/content/data/ultrasound-nerve/ultrasound-nerve-segmentation/sample_submission.csv",
  "image_column": "image",
  "mask_column": "mask",
  "label_column": "label",
  "group_column": "patient_id",
  "image_path_template": "{image}",
  "mask_path_template": "{mask}",
  "image_extension": "",
  "mask_extension": "",
  "input_channels": 1,
  "output_channels": 1,
  "num_

In [16]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "unet",
      "finetune_strategy": "full",
      "loss": "bce_dice_loss",
      "optimizer": "adam",
      "rank": 1,
      "score": 0.814,
      "task_type": "image_segmentation"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "bce_dice_loss",
      "optimizer": "adamw",
      "rank": 2,
      "score": 0.629,
      "task_type": "image_segmentation"
    },
    {
      "backbone": "swin_transformer",
      "finetune_strategy": "full",
      "loss": "bce_dice_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.584,
      "task_type": "image_segmentation"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "evaluate.py",
    "generation_info.json",
    "infer.py",
    "model.py",
    "model_utils.py",
    "module4_summary.json",
    "requirements.txt",
    "run.py",
    "run_experiments.py",
    "smoke_data.py",
    "train.py",
    "uti

In [17]:
# Preflight check for Ultrasound Nerve Segmentation

import json
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/jiaozi_generated_pipeline"
)

config_path = (
    OUTPUT_DIR / "configs.json"
)

if not config_path.is_file():
    raise FileNotFoundError(
        f"configs.json not found: {config_path}"
    )

configs = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

if not configs:
    raise RuntimeError(
        "configs.json is empty."
    )

cfg = configs[0]

flat_cfg = dict(cfg)

if isinstance(
    cfg.get("model_config"),
    dict,
):
    flat_cfg.update(
        cfg["model_config"]
    )

task_type = str(
    flat_cfg.get(
        "task_type",
        "",
    )
).lower()

problem_type = str(
    flat_cfg.get(
        "problem_type",
        "",
    )
).lower()

metric = str(
    flat_cfg.get(
        "evaluation_metric",
        flat_cfg.get(
            "metric_name",
            flat_cfg.get(
                "metric",
                "",
            ),
        ),
    )
).lower()

image_dir = flat_cfg.get(
    "image_dir"
)

mask_dir = flat_cfg.get(
    "mask_dir"
)

mask_column = flat_cfg.get(
    "mask_column"
)

print(
    "=== MODULE 4 PREFLIGHT CHECK ==="
)

print("task_type        :", task_type)
print("problem_type     :", problem_type)
print("evaluation_metric:", metric)
print(
    "input_channels   :",
    flat_cfg.get("input_channels"),
)
print(
    "output_channels  :",
    flat_cfg.get("output_channels"),
)
print("image_dir        :", image_dir)
print("mask_dir         :", mask_dir)
print("mask_column      :", mask_column)

segmentation_ok = (
    "segmentation" in task_type
    or "segmentation" in problem_type
)

metric_ok = metric in {
    "dice",
    "dice_coefficient",
    "dice_score",
}

image_dir_ok = (
    image_dir is not None
    and Path(str(image_dir)).is_dir()
)

mask_dir_ok = (
    mask_dir is not None
    and Path(str(mask_dir)).is_dir()
)

mask_column_ok = (
    str(mask_column).strip()
    == "mask"
)

failed = []

if not segmentation_ok:
    failed.append(
        "task is not segmentation"
    )

if not metric_ok:
    failed.append(
        "metric is not Dice"
    )

if not image_dir_ok:
    failed.append(
        "image_dir is invalid"
    )

if not mask_dir_ok:
    failed.append(
        "mask_dir is invalid"
    )

if not mask_column_ok:
    failed.append(
        "mask_column is invalid"
    )

if failed:
    raise RuntimeError(
        "Module 4 preflight failed:\n- "
        + "\n- ".join(failed)
        + "\n\nDo not start training."
    )

# Search generated source for obvious classification fallback.
source_text = ""

for filename in [
    "model.py",
    "train.py",
    "evaluate.py",
    "predict.py",
]:
    path = OUTPUT_DIR / filename

    if path.is_file():
        source_text += path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).lower()

classification_markers = [
    "class_id",
    "macro_f1",
    "val_acc",
    "classificationmodel",
]

segmentation_markers = [
    "mask",
    "dice",
    "segmentation",
]

found_bad = [
    marker
    for marker in classification_markers
    if marker in source_text
]

found_good = [
    marker
    for marker in segmentation_markers
    if marker in source_text
]

print(
    "Segmentation markers:",
    found_good,
)

print(
    "Classification markers:",
    found_bad,
)

if not found_good:
    raise RuntimeError(
        "Generated files contain no clear "
        "segmentation or Dice implementation."
    )

if (
    "classificationmodel" in found_bad
    or "class_id" in found_bad
):
    raise RuntimeError(
        "Generated project appears to have "
        "fallen back to classification. "
        "Do not start training."
    )

print(
    "\n✅ Module 4 generated a valid "
    "segmentation-oriented configuration."
)


=== MODULE 4 PREFLIGHT CHECK ===
task_type        : image_segmentation
problem_type     : binary_semantic_segmentation
evaluation_metric: dice
input_channels   : 1
output_channels  : 1
image_dir        : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_images
mask_dir         : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_masks
mask_column      : mask
Segmentation markers: ['mask', 'dice', 'segmentation']
Classification markers: ['macro_f1', 'val_acc']

✅ Module 4 generated a valid segmentation-oriented configuration.


## Train the recommended model (real data)

This step only runs when `REAL_TRAINING = True`. It executes the generated `run.py`,
which loads the real dataset, trains the model Module 3 recommended, evaluates on the
test split, and saves checkpoints. Select a GPU runtime first for reasonable speed.


In [18]:
# Train the RAG-recommended Ultrasound Nerve Segmentation model
#
# The Module 1 -> Module 2 -> Module 3 -> Module 4 workflow remains unchanged.
# This Cell reads the architecture and hyperparameters recommended by the
# pipeline. It only replaces the invalid classification-oriented trainer
# generated by the generic Module 4 template.

import json
import math
import random
import shutil
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


# ============================================================
# 1. Load Module 4 recommendation
# ============================================================

OUTPUT_DIR = Path(
    "/content/jiaozi_generated_pipeline"
)

config_path = (
    OUTPUT_DIR / "configs.json"
)

if not config_path.is_file():
    raise FileNotFoundError(
        f"configs.json not found: {config_path}\n"
        "Run Modules 1-4 first."
    )

configs = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

if not configs:
    raise RuntimeError(
        "configs.json is empty."
    )

cfg = configs[0]
model_cfg = cfg.get("model_config", {})

if not isinstance(model_cfg, dict):
    model_cfg = {}

# model_config takes priority over outer generic fields.
recommended_cfg = {
    **cfg,
    **model_cfg,
}


def first_available(
    mapping,
    names,
    default,
):
    for name in names:
        value = mapping.get(name)

        if value is not None:
            return value

    return default


def parse_image_size(value):
    if isinstance(value, int):
        return int(value), int(value)

    if (
        isinstance(value, (list, tuple))
        and len(value) >= 2
    ):
        return (
            int(value[-2]),
            int(value[-1]),
        )

    if isinstance(value, str):
        cleaned = (
            value.lower()
            .replace("×", "x")
            .replace(" ", "")
        )

        if "x" in cleaned:
            parts = cleaned.split("x")

            if len(parts) >= 2:
                try:
                    return (
                        int(parts[-2]),
                        int(parts[-1]),
                    )
                except ValueError:
                    pass

        try:
            scalar = int(cleaned)
            return scalar, scalar
        except ValueError:
            pass

    return 256, 256


RECOMMENDED_ARCHITECTURE = str(
    first_available(
        recommended_cfg,
        [
            "architecture",
            "model_name",
            "backbone",
            "architecture_family",
        ],
        "unet",
    )
)

EPOCHS = int(
    first_available(
        recommended_cfg,
        [
            "recommended_epochs",
            "epochs",
            "num_epochs",
            "max_epochs",
        ],
        30,
    )
)

BATCH_SIZE = int(
    first_available(
        recommended_cfg,
        [
            "batch_size",
            "train_batch_size",
        ],
        8,
    )
)

LEARNING_RATE = float(
    first_available(
        recommended_cfg,
        [
            "learning_rate",
            "lr",
        ],
        1e-3,
    )
)

WEIGHT_DECAY = float(
    first_available(
        recommended_cfg,
        [
            "weight_decay",
            "l2_regularization",
        ],
        1e-5,
    )
)

EARLY_STOPPING_PATIENCE = int(
    first_available(
        recommended_cfg,
        [
            "early_stopping_patience",
            "patience",
        ],
        8,
    )
)

PREDICTION_THRESHOLD = float(
    first_available(
        recommended_cfg,
        [
            "prediction_threshold",
            "threshold",
            "mask_threshold",
        ],
        0.5,
    )
)

BASE_CHANNELS = int(
    first_available(
        recommended_cfg,
        [
            "base_channels",
            "initial_channels",
            "features",
        ],
        16,
    )
)

IMAGE_HEIGHT, IMAGE_WIDTH = parse_image_size(
    first_available(
        recommended_cfg,
        [
            "image_size",
            "input_size",
            "resize",
            "resolution",
        ],
        256,
    )
)

OPTIMIZER_NAME = str(
    first_available(
        recommended_cfg,
        ["optimizer"],
        "adamw",
    )
).lower()

LOSS_NAME = str(
    first_available(
        recommended_cfg,
        [
            "loss_function",
            "loss",
        ],
        "bce_dice_loss",
    )
).lower()

SEED = int(
    first_available(
        recommended_cfg,
        ["seed", "random_seed"],
        42,
    )
)

NUM_WORKERS = int(
    first_available(
        recommended_cfg,
        ["num_workers"],
        2,
    )
)

print(
    "=== PIPELINE RECOMMENDATION USED FOR TRAINING ==="
)

print(
    "architecture             :",
    RECOMMENDED_ARCHITECTURE,
)

print(
    "recommended_epochs       :",
    EPOCHS,
)

print(
    "batch_size               :",
    BATCH_SIZE,
)

print(
    "learning_rate            :",
    LEARNING_RATE,
)

print(
    "weight_decay             :",
    WEIGHT_DECAY,
)

print(
    "image_size               :",
    (IMAGE_HEIGHT, IMAGE_WIDTH),
)

print(
    "optimizer                :",
    OPTIMIZER_NAME,
)

print(
    "loss                     :",
    LOSS_NAME,
)

print(
    "early_stopping_patience  :",
    EARLY_STOPPING_PATIENCE,
)

print(
    "prediction_threshold     :",
    PREDICTION_THRESHOLD,
)

print(
    "base_channels            :",
    BASE_CHANNELS,
)


# ============================================================
# 2. Validate the recommended task
# ============================================================

task_type = str(
    recommended_cfg.get(
        "task_type",
        "",
    )
).lower()

problem_type = str(
    recommended_cfg.get(
        "problem_type",
        "",
    )
).lower()

metric_name = str(
    first_available(
        recommended_cfg,
        [
            "evaluation_metric",
            "metric_name",
            "metric",
        ],
        "",
    )
).lower()

if (
    "segmentation" not in task_type
    and "segmentation" not in problem_type
):
    raise RuntimeError(
        "The pipeline did not recommend a "
        "segmentation task. Do not train."
    )

if metric_name not in {
    "dice",
    "dice_score",
    "dice_coefficient",
}:
    raise RuntimeError(
        "The recommended metric is not Dice: "
        f"{metric_name!r}"
    )

architecture_lower = (
    RECOMMENDED_ARCHITECTURE.lower()
)

supported_architecture = any(
    keyword in architecture_lower
    for keyword in [
        "unet",
        "u-net",
        "encoder_decoder",
        "encoder-decoder",
        "segmentation",
        "fcn",
    ]
)

if not supported_architecture:
    raise RuntimeError(
        "The RAG recommended an architecture that "
        "this Notebook segmentation adapter does not "
        "currently implement:\n"
        f"{RECOMMENDED_ARCHITECTURE}\n"
        "Do not silently replace the recommendation."
    )


# ============================================================
# 3. Reproducibility and paths
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

manifest_path = Path(
    recommended_cfg.get(
        "train_csv",
        processed_train_csv,
    )
)

image_root = Path(
    recommended_cfg.get(
        "image_dir",
        TRAIN_IMAGE_DIR,
    )
)

mask_root = Path(
    recommended_cfg.get(
        "mask_dir",
        TRAIN_MASK_DIR,
    )
)

if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Manifest not found: {manifest_path}"
    )

if not image_root.is_dir():
    raise FileNotFoundError(
        f"Image directory not found: {image_root}"
    )

if not mask_root.is_dir():
    raise FileNotFoundError(
        f"Mask directory not found: {mask_root}"
    )

checkpoint_dir = Path(
    "/content/drive/MyDrive/"
    "Jiaozi/checkpoints/"
    "ultrasound_nerve_dice"
)

important_dir = (
    checkpoint_dir
    / "important_epochs"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

important_dir.mkdir(
    parents=True,
    exist_ok=True,
)

best_model_path = (
    checkpoint_dir / "best_model.pt"
)

training_log_path = (
    checkpoint_dir / "training.log"
)

metrics_path = (
    checkpoint_dir
    / "epoch_metrics.jsonl"
)

print("\nDevice ->", device)
print("Manifest ->", manifest_path)
print("Images ->", image_root)
print("Masks ->", mask_root)
print("Checkpoints ->", checkpoint_dir)
print(
    "Representative checkpoints ->",
    important_dir,
)
print("Resume checkpoint -> disabled")
print("evaluation_metric -> dice")


# ============================================================
# 4. Patient-level split
# ============================================================

frame = pd.read_csv(
    manifest_path
)

required_columns = {
    "image",
    "mask",
    "patient_id",
}

missing_columns = (
    required_columns
    - set(frame.columns)
)

if missing_columns:
    raise ValueError(
        "Manifest is missing columns: "
        f"{sorted(missing_columns)}"
    )

frame["patient_id"] = (
    frame["patient_id"].astype(str)
)

patients = sorted(
    frame["patient_id"]
    .unique()
    .tolist()
)

split_rng = random.Random(SEED)
split_rng.shuffle(patients)

validation_fraction = float(
    first_available(
        recommended_cfg,
        [
            "validation_split",
            "val_split",
            "validation_fraction",
        ],
        0.2,
    )
)

validation_fraction = min(
    max(validation_fraction, 0.05),
    0.5,
)

num_val_patients = max(
    1,
    int(
        round(
            len(patients)
            * validation_fraction
        )
    ),
)

val_patients = set(
    patients[:num_val_patients]
)

train_frame = frame[
    ~frame["patient_id"].isin(
        val_patients
    )
].reset_index(drop=True)

val_frame = frame[
    frame["patient_id"].isin(
        val_patients
    )
].reset_index(drop=True)

if train_frame.empty or val_frame.empty:
    raise RuntimeError(
        "Patient-level split created an empty set."
    )

print("\nPatient-level split:")
print("Total patients :", len(patients))
print(
    "Train patients :",
    train_frame[
        "patient_id"
    ].nunique(),
)
print(
    "Val patients   :",
    val_frame[
        "patient_id"
    ].nunique(),
)
print("Train images   :", len(train_frame))
print("Val images     :", len(val_frame))


# ============================================================
# 5. Dataset
# ============================================================

class UltrasoundSegmentationDataset(
    Dataset
):
    def __init__(
        self,
        data_frame,
        image_dir,
        mask_dir,
        training,
        output_size,
    ):
        self.frame = (
            data_frame
            .reset_index(drop=True)
        )

        self.image_dir = Path(
            image_dir
        )

        self.mask_dir = Path(
            mask_dir
        )

        self.training = bool(
            training
        )

        self.output_height = int(
            output_size[0]
        )

        self.output_width = int(
            output_size[1]
        )

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[
            index
        ]

        image_path = (
            self.image_dir
            / str(row["image"])
        )

        mask_path = (
            self.mask_dir
            / str(row["mask"])
        )

        if not image_path.is_file():
            raise FileNotFoundError(
                image_path
            )

        if not mask_path.is_file():
            raise FileNotFoundError(
                mask_path
            )

        with Image.open(
            image_path
        ) as image:
            image = image.convert(
                "L"
            )

            image = image.resize(
                (
                    self.output_width,
                    self.output_height,
                ),
                resample=Image.BILINEAR,
            )

            image_array = (
                np.asarray(
                    image,
                    dtype=np.float32,
                )
                / 255.0
            )

        with Image.open(
            mask_path
        ) as mask:
            mask = mask.convert(
                "L"
            )

            mask = mask.resize(
                (
                    self.output_width,
                    self.output_height,
                ),
                resample=Image.NEAREST,
            )

            mask_array = (
                np.asarray(
                    mask,
                    dtype=np.uint8,
                )
                > 0
            ).astype(np.float32)

        image_tensor = (
            torch.from_numpy(
                image_array
            )
            .unsqueeze(0)
        )

        mask_tensor = (
            torch.from_numpy(
                mask_array
            )
            .unsqueeze(0)
        )

        if self.training:
            if random.random() < 0.5:
                image_tensor = torch.flip(
                    image_tensor,
                    dims=[2],
                )

                mask_tensor = torch.flip(
                    mask_tensor,
                    dims=[2],
                )

            if random.random() < 0.5:
                image_tensor = torch.flip(
                    image_tensor,
                    dims=[1],
                )

                mask_tensor = torch.flip(
                    mask_tensor,
                    dims=[1],
                )

            rotation_k = random.randint(
                0,
                3,
            )

            if rotation_k:
                image_tensor = torch.rot90(
                    image_tensor,
                    k=rotation_k,
                    dims=[1, 2],
                )

                mask_tensor = torch.rot90(
                    mask_tensor,
                    k=rotation_k,
                    dims=[1, 2],
                )

            if random.random() < 0.35:
                gain = random.uniform(
                    0.9,
                    1.1,
                )

                bias = random.uniform(
                    -0.04,
                    0.04,
                )

                image_tensor = (
                    image_tensor * gain
                    + bias
                ).clamp(
                    0.0,
                    1.0,
                )

        return {
            "image": (
                image_tensor.float()
            ),
            "mask": (
                mask_tensor.float()
            ),
        }


train_dataset = (
    UltrasoundSegmentationDataset(
        train_frame,
        image_root,
        mask_root,
        training=True,
        output_size=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
    )
)

val_dataset = (
    UltrasoundSegmentationDataset(
        val_frame,
        image_root,
        mask_root,
        training=False,
        output_size=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
    )
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(
        device.type == "cuda"
    ),
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        device.type == "cuda"
    ),
    drop_last=False,
)


# ============================================================
# 6. U-Net implementation for the recommended family
# ============================================================

class DoubleConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(
                in_channels,
                out_channels,
            ),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.conv = DoubleConv(
            out_channels
            + skip_channels,
            out_channels,
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = self.up(x)

        if x.shape[-2:] != (
            skip.shape[-2:]
        ):
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        return self.conv(
            torch.cat(
                [skip, x],
                dim=1,
            )
        )


class RecommendedUNet(nn.Module):
    def __init__(
        self,
        input_channels=1,
        output_channels=1,
        base_channels=16,
    ):
        super().__init__()

        self.encoder1 = DoubleConv(
            input_channels,
            base_channels,
        )

        self.encoder2 = DownBlock(
            base_channels,
            base_channels * 2,
        )

        self.encoder3 = DownBlock(
            base_channels * 2,
            base_channels * 4,
        )

        self.bottleneck = DownBlock(
            base_channels * 4,
            base_channels * 8,
        )

        self.decoder3 = UpBlock(
            base_channels * 8,
            base_channels * 4,
            base_channels * 4,
        )

        self.decoder2 = UpBlock(
            base_channels * 4,
            base_channels * 2,
            base_channels * 2,
        )

        self.decoder1 = UpBlock(
            base_channels * 2,
            base_channels,
            base_channels,
        )

        self.output_head = nn.Conv2d(
            base_channels,
            output_channels,
            kernel_size=1,
        )

    def forward(self, x):
        skip1 = self.encoder1(x)
        skip2 = self.encoder2(skip1)
        skip3 = self.encoder3(skip2)

        bottleneck = self.bottleneck(
            skip3
        )

        x = self.decoder3(
            bottleneck,
            skip3,
        )

        x = self.decoder2(
            x,
            skip2,
        )

        x = self.decoder1(
            x,
            skip1,
        )

        return self.output_head(x)


model = RecommendedUNet(
    input_channels=1,
    output_channels=1,
    base_channels=BASE_CHANNELS,
).to(device)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "\nRecommended architecture:",
    RECOMMENDED_ARCHITECTURE,
)

print(
    "Notebook implementation:",
    "RecommendedUNet",
)

print(
    "Total parameters:",
    total_params,
)

print(
    "Trainable parameters:",
    trainable_params,
)

if total_params < 100_000:
    raise RuntimeError(
        "The segmentation model is "
        "unexpectedly small."
    )


# ============================================================
# 7. Recommended loss
# ============================================================

bce_loss_function = (
    nn.BCEWithLogitsLoss()
)


def soft_dice_loss(
    logits,
    targets,
    epsilon=1e-6,
):
    probabilities = (
        torch.sigmoid(logits)
        .flatten(start_dim=1)
    )

    targets = targets.flatten(
        start_dim=1
    )

    intersection = (
        probabilities * targets
    ).sum(dim=1)

    denominator = (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
    )

    dice = (
        2.0 * intersection
        + epsilon
    ) / (
        denominator + epsilon
    )

    return 1.0 - dice.mean()


def segmentation_loss(
    logits,
    targets,
):
    loss_lower = LOSS_NAME.lower()

    if (
        "bce" in loss_lower
        and "dice" in loss_lower
    ):
        return (
            0.5
            * bce_loss_function(
                logits,
                targets,
            )
            + 0.5
            * soft_dice_loss(
                logits,
                targets,
            )
        )

    if "dice" in loss_lower:
        return soft_dice_loss(
            logits,
            targets,
        )

    if "focal" in loss_lower:
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        probability = torch.sigmoid(
            logits
        )

        pt = (
            probability * targets
            + (1.0 - probability)
            * (1.0 - targets)
        )

        focal = (
            ((1.0 - pt) ** 2.0)
            * bce
        ).mean()

        return (
            0.5 * focal
            + 0.5
            * soft_dice_loss(
                logits,
                targets,
            )
        )

    # Safe segmentation fallback, never classification CE.
    return (
        0.5
        * bce_loss_function(
            logits,
            targets,
        )
        + 0.5
        * soft_dice_loss(
            logits,
            targets,
        )
    )


def dice_scores(
    logits,
    targets,
    threshold,
    epsilon=1e-6,
):
    predictions = (
        torch.sigmoid(logits)
        >= threshold
    ).float()

    predictions = predictions.flatten(
        start_dim=1
    )

    targets = targets.flatten(
        start_dim=1
    )

    intersection = (
        predictions * targets
    ).sum(dim=1)

    denominator = (
        predictions.sum(dim=1)
        + targets.sum(dim=1)
    )

    scores = (
        2.0 * intersection
        + epsilon
    ) / (
        denominator + epsilon
    )

    both_empty = (
        denominator == 0
    )

    return torch.where(
        both_empty,
        torch.ones_like(scores),
        scores,
    )


# ============================================================
# 8. Recommended optimizer
# ============================================================

if OPTIMIZER_NAME == "adam":
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

elif OPTIMIZER_NAME == "sgd":
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,
    )

else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

scheduler = (
    torch.optim.lr_scheduler
    .CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-6,
    )
)

use_amp = (
    device.type == "cuda"
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)


# ============================================================
# 9. Validation
# ============================================================

def evaluate_model():
    model.eval()

    total_loss = 0.0
    total_samples = 0
    scores = []

    with torch.inference_mode():
        for batch in val_loader:
            images = batch["image"].to(
                device,
                non_blocking=True,
            )

            masks = batch["mask"].to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(images)

                loss = segmentation_loss(
                    logits,
                    masks,
                )

            batch_size = images.shape[0]

            total_loss += (
                float(loss.item())
                * batch_size
            )

            total_samples += batch_size

            scores.extend(
                dice_scores(
                    logits.float(),
                    masks,
                    PREDICTION_THRESHOLD,
                )
                .cpu()
                .tolist()
            )

    return {
        "loss": (
            total_loss
            / max(total_samples, 1)
        ),
        "dice": float(
            np.mean(scores)
        ),
        "num_samples": total_samples,
    }


# ============================================================
# 10. Train using recommended settings
# ============================================================

training_log_path.write_text(
    "",
    encoding="utf-8",
)

metrics_path.write_text(
    "",
    encoding="utf-8",
)

best_dice = -math.inf
best_epoch = None
epochs_without_improvement = 0
training_start = time.time()

print(
    f"\nTraining {RECOMMENDED_ARCHITECTURE} "
    f"for {EPOCHS} recommended epochs..."
)

with training_log_path.open(
    "a",
    encoding="utf-8",
    buffering=1,
) as log_file:

    log_file.write(
        "===== RUN START "
        f"{datetime.now().isoformat()} "
        "=====\n"
    )

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        epoch_start = time.time()
        model.train()

        running_loss = 0.0
        seen_samples = 0

        for batch in train_loader:
            images = batch["image"].to(
                device,
                non_blocking=True,
            )

            masks = batch["mask"].to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(images)

                if logits.shape != masks.shape:
                    raise RuntimeError(
                        "Segmentation output and mask "
                        "shapes do not match:\n"
                        f"output={tuple(logits.shape)}\n"
                        f"mask={tuple(masks.shape)}"
                    )

                loss = segmentation_loss(
                    logits,
                    masks,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            batch_size = images.shape[0]

            running_loss += (
                float(loss.item())
                * batch_size
            )

            seen_samples += batch_size

        scheduler.step()

        train_loss = (
            running_loss
            / max(seen_samples, 1)
        )

        validation = evaluate_model()

        val_loss = float(
            validation["loss"]
        )

        val_dice = float(
            validation["dice"]
        )

        current_lr = float(
            optimizer.param_groups[0]["lr"]
        )

        epoch_time = (
            time.time()
            - epoch_start
        )

        line = (
            f"[train] epoch {epoch}/{EPOCHS}  "
            f"loss={train_loss:.6f}  "
            f"val_loss={val_loss:.6f}  "
            f"val_dice={val_dice:.6f}  "
            f"lr={current_lr:.2e}  "
            f"steps={len(train_loader)}  "
            f"time={epoch_time:.1f}s"
        )

        print(
            line,
            flush=True,
        )

        log_file.write(
            line + "\n"
        )

        record = {
            "epoch": epoch,
            "total_epochs": EPOCHS,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_dice": val_dice,
            "learning_rate": current_lr,
            "recommended_architecture": (
                RECOMMENDED_ARCHITECTURE
            ),
            "recommended_loss": LOSS_NAME,
            "recommended_optimizer": (
                OPTIMIZER_NAME
            ),
            "timestamp": (
                datetime.now()
                .isoformat()
            ),
        }

        with metrics_path.open(
            "a",
            encoding="utf-8",
        ) as metrics_file:
            metrics_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch
            epochs_without_improvement = 0

            checkpoint = {
                "model_state_dict": (
                    model.state_dict()
                ),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),
                "best_epoch": best_epoch,
                "best_metric": best_dice,
                "validation": {
                    "metric_name": "dice",
                    "metric_value": best_dice,
                    "dice": best_dice,
                    "loss": val_loss,
                    "num_samples": int(
                        validation[
                            "num_samples"
                        ]
                    ),
                    "empty_mask_policy": (
                        "both empty equals Dice 1"
                    ),
                },
                "config": {
                    **recommended_cfg,
                    "recommended_architecture": (
                        RECOMMENDED_ARCHITECTURE
                    ),
                    "implemented_model_class": (
                        "RecommendedUNet"
                    ),
                    "recommended_epochs": EPOCHS,
                    "batch_size": BATCH_SIZE,
                    "learning_rate": (
                        LEARNING_RATE
                    ),
                    "weight_decay": (
                        WEIGHT_DECAY
                    ),
                    "image_height": (
                        IMAGE_HEIGHT
                    ),
                    "image_width": (
                        IMAGE_WIDTH
                    ),
                    "base_channels": (
                        BASE_CHANNELS
                    ),
                    "prediction_threshold": (
                        PREDICTION_THRESHOLD
                    ),
                    "loss": LOSS_NAME,
                    "optimizer": (
                        OPTIMIZER_NAME
                    ),
                    "validation_patients": (
                        sorted(val_patients)
                    ),
                },
                "model_class": (
                    "RecommendedUNet"
                ),
                "task_type": (
                    "image_segmentation"
                ),
                "total_params": total_params,
                "trainable_params": (
                    trainable_params
                ),
            }

            torch.save(
                checkpoint,
                best_model_path,
            )

            representative_path = (
                important_dir
                / (
                    "best_epoch_"
                    f"{epoch:03d}.pt"
                )
            )

            shutil.copy2(
                best_model_path,
                representative_path,
            )

            print(
                "[train] Saved new best "
                f"checkpoint at epoch {epoch}: "
                f"{best_model_path}",
                flush=True,
            )

            print(
                "[pipeline] Preserved "
                "representative checkpoint:",
                representative_path,
                flush=True,
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            message = (
                "[train] Early stopping after "
                f"{EARLY_STOPPING_PATIENCE} epochs "
                "without validation improvement."
            )

            print(
                message,
                flush=True,
            )

            log_file.write(
                message + "\n"
            )

            break

    log_file.write(
        "===== RUN END "
        f"{datetime.now().isoformat()} "
        "=====\n"
    )


# ============================================================
# 11. Final result
# ============================================================

if not best_model_path.is_file():
    raise RuntimeError(
        "Training did not produce best_model.pt."
    )

print(
    "\nReal segmentation training finished."
)

print(
    "RAG-recommended architecture:",
    RECOMMENDED_ARCHITECTURE,
)

print(
    "Recommended epochs:",
    EPOCHS,
)

print(
    "Best checkpoint:",
    best_model_path,
)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation Dice:",
    best_dice,
)

print(
    "Representative checkpoints:",
    important_dir,
)

print(
    "Epoch metrics:",
    metrics_path,
)

print(
    "Training log:",
    training_log_path,
)

print(
    "Total runtime:",
    round(
        time.time()
        - training_start,
        1,
    ),
    "seconds",
)


=== PIPELINE RECOMMENDATION USED FOR TRAINING ===
architecture             : unet
recommended_epochs       : 30
batch_size               : 8
learning_rate            : 0.001
weight_decay             : 1e-05
image_size               : (224, 224)
optimizer                : adam
loss                     : bce_dice_loss
early_stopping_patience  : 8
prediction_threshold     : 0.5
base_channels            : 16

Device -> cuda
Manifest -> /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_ultrasound_segmentation.csv
Images -> /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_images
Masks -> /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_masks
Checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/ultrasound_nerve_dice
Representative checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/ultrasound_nerve_dice/important_epochs
Resume checkpoint -> disabled
evaluation_metric -> dice

Patient-level split:
Total patients : 47


In [19]:
# Continue training the RAG-recommended Ultrasound Nerve Segmentation model
# from the existing best checkpoint until the recommended total epoch count.
#
# Pipeline-recommended architecture and hyperparameters remain unchanged.
# Early-stopping status is logged, but training is not stopped before EPOCHS.

import json
import math
import random
import shutil
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


# ============================================================
# 1. Load Module 4 recommendation
# ============================================================

OUTPUT_DIR = Path(
    "/content/jiaozi_generated_pipeline"
)

config_path = OUTPUT_DIR / "configs.json"

if not config_path.is_file():
    raise FileNotFoundError(
        f"configs.json not found: {config_path}\n"
        "Run Modules 1-4 first."
    )

configs = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

if not configs:
    raise RuntimeError(
        "configs.json is empty."
    )

cfg = configs[0]

model_cfg = cfg.get(
    "model_config",
    {},
)

if not isinstance(
    model_cfg,
    dict,
):
    model_cfg = {}

recommended_cfg = {
    **cfg,
    **model_cfg,
}


def first_available(
    mapping,
    names,
    default,
):
    for name in names:
        value = mapping.get(name)

        if value is not None:
            return value

    return default


def parse_image_size(value):
    if isinstance(value, int):
        return int(value), int(value)

    if (
        isinstance(value, (list, tuple))
        and len(value) >= 2
    ):
        return (
            int(value[-2]),
            int(value[-1]),
        )

    if isinstance(value, str):
        cleaned = (
            value.lower()
            .replace("×", "x")
            .replace(" ", "")
        )

        if "x" in cleaned:
            parts = cleaned.split("x")

            if len(parts) >= 2:
                try:
                    return (
                        int(parts[-2]),
                        int(parts[-1]),
                    )
                except ValueError:
                    pass

        try:
            scalar = int(cleaned)
            return scalar, scalar
        except ValueError:
            pass

    return 256, 256


RECOMMENDED_ARCHITECTURE = str(
    first_available(
        recommended_cfg,
        [
            "architecture",
            "model_name",
            "backbone",
            "architecture_family",
        ],
        "unet",
    )
)

EPOCHS = int(
    first_available(
        recommended_cfg,
        [
            "recommended_epochs",
            "epochs",
            "num_epochs",
            "max_epochs",
        ],
        30,
    )
)

BATCH_SIZE = int(
    first_available(
        recommended_cfg,
        [
            "batch_size",
            "train_batch_size",
        ],
        8,
    )
)

LEARNING_RATE = float(
    first_available(
        recommended_cfg,
        [
            "learning_rate",
            "lr",
        ],
        1e-3,
    )
)

WEIGHT_DECAY = float(
    first_available(
        recommended_cfg,
        [
            "weight_decay",
            "l2_regularization",
        ],
        1e-5,
    )
)

EARLY_STOPPING_PATIENCE = int(
    first_available(
        recommended_cfg,
        [
            "early_stopping_patience",
            "patience",
        ],
        8,
    )
)

PREDICTION_THRESHOLD = float(
    first_available(
        recommended_cfg,
        [
            "prediction_threshold",
            "threshold",
            "mask_threshold",
        ],
        0.5,
    )
)

BASE_CHANNELS = int(
    first_available(
        recommended_cfg,
        [
            "base_channels",
            "initial_channels",
            "features",
        ],
        16,
    )
)

IMAGE_HEIGHT, IMAGE_WIDTH = parse_image_size(
    first_available(
        recommended_cfg,
        [
            "image_size",
            "input_size",
            "resize",
            "resolution",
        ],
        256,
    )
)

OPTIMIZER_NAME = str(
    first_available(
        recommended_cfg,
        ["optimizer"],
        "adamw",
    )
).lower()

LOSS_NAME = str(
    first_available(
        recommended_cfg,
        [
            "loss_function",
            "loss",
        ],
        "bce_dice_loss",
    )
).lower()

SEED = int(
    first_available(
        recommended_cfg,
        [
            "seed",
            "random_seed",
        ],
        42,
    )
)

NUM_WORKERS = int(
    first_available(
        recommended_cfg,
        ["num_workers"],
        2,
    )
)

VALIDATION_FRACTION = float(
    first_available(
        recommended_cfg,
        [
            "validation_split",
            "val_split",
            "validation_fraction",
        ],
        0.2,
    )
)

VALIDATION_FRACTION = min(
    max(
        VALIDATION_FRACTION,
        0.05,
    ),
    0.5,
)

print(
    "=== PIPELINE RECOMMENDATION USED FOR TRAINING ==="
)

print(
    "architecture            :",
    RECOMMENDED_ARCHITECTURE,
)
print(
    "recommended_epochs      :",
    EPOCHS,
)
print(
    "batch_size              :",
    BATCH_SIZE,
)
print(
    "learning_rate           :",
    LEARNING_RATE,
)
print(
    "weight_decay            :",
    WEIGHT_DECAY,
)
print(
    "image_size              :",
    (IMAGE_HEIGHT, IMAGE_WIDTH),
)
print(
    "optimizer               :",
    OPTIMIZER_NAME,
)
print(
    "loss                    :",
    LOSS_NAME,
)
print(
    "early_stopping_patience :",
    EARLY_STOPPING_PATIENCE,
)
print(
    "prediction_threshold    :",
    PREDICTION_THRESHOLD,
)
print(
    "base_channels           :",
    BASE_CHANNELS,
)
print(
    "validation_fraction     :",
    VALIDATION_FRACTION,
)


# ============================================================
# 2. Validate recommended task
# ============================================================

task_type = str(
    recommended_cfg.get(
        "task_type",
        "",
    )
).lower()

problem_type = str(
    recommended_cfg.get(
        "problem_type",
        "",
    )
).lower()

metric_name = str(
    first_available(
        recommended_cfg,
        [
            "evaluation_metric",
            "metric_name",
            "metric",
        ],
        "",
    )
).lower()

if (
    "segmentation" not in task_type
    and "segmentation" not in problem_type
):
    raise RuntimeError(
        "The pipeline did not recommend a "
        "segmentation task."
    )

if metric_name not in {
    "dice",
    "dice_score",
    "dice_coefficient",
}:
    raise RuntimeError(
        "The recommended metric is not Dice: "
        f"{metric_name!r}"
    )

architecture_lower = (
    RECOMMENDED_ARCHITECTURE.lower()
)

supported_architecture = any(
    keyword in architecture_lower
    for keyword in [
        "unet",
        "u-net",
        "encoder_decoder",
        "encoder-decoder",
        "segmentation",
        "fcn",
    ]
)

if not supported_architecture:
    raise RuntimeError(
        "Unsupported recommended segmentation "
        "architecture:\n"
        f"{RECOMMENDED_ARCHITECTURE}"
    )


# ============================================================
# 3. Reproducibility and paths
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

manifest_path = Path(
    recommended_cfg.get(
        "train_csv",
        processed_train_csv,
    )
)

image_root = Path(
    recommended_cfg.get(
        "image_dir",
        TRAIN_IMAGE_DIR,
    )
)

mask_root = Path(
    recommended_cfg.get(
        "mask_dir",
        TRAIN_MASK_DIR,
    )
)

if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Manifest not found: {manifest_path}"
    )

if not image_root.is_dir():
    raise FileNotFoundError(
        f"Image directory not found: {image_root}"
    )

if not mask_root.is_dir():
    raise FileNotFoundError(
        f"Mask directory not found: {mask_root}"
    )

checkpoint_dir = Path(
    "/content/drive/MyDrive/"
    "Jiaozi/checkpoints/"
    "ultrasound_nerve_dice"
)

important_dir = (
    checkpoint_dir
    / "important_epochs"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

important_dir.mkdir(
    parents=True,
    exist_ok=True,
)

best_model_path = (
    checkpoint_dir
    / "best_model.pt"
)

training_log_path = (
    checkpoint_dir
    / "training.log"
)

metrics_path = (
    checkpoint_dir
    / "epoch_metrics.jsonl"
)

print("\nDevice ->", device)
print("Manifest ->", manifest_path)
print("Images ->", image_root)
print("Masks ->", mask_root)
print("Checkpoints ->", checkpoint_dir)
print(
    "Representative checkpoints ->",
    important_dir,
)
print("Resume checkpoint -> enabled")
print("evaluation_metric -> dice")


# ============================================================
# 4. Patient-level split
# ============================================================

frame = pd.read_csv(
    manifest_path
)

required_columns = {
    "image",
    "mask",
    "patient_id",
}

missing_columns = (
    required_columns
    - set(frame.columns)
)

if missing_columns:
    raise ValueError(
        "Manifest is missing columns: "
        f"{sorted(missing_columns)}"
    )

frame["patient_id"] = (
    frame["patient_id"]
    .astype(str)
)

patients = sorted(
    frame["patient_id"]
    .unique()
    .tolist()
)

split_rng = random.Random(
    SEED
)

split_rng.shuffle(
    patients
)

num_val_patients = max(
    1,
    int(
        round(
            len(patients)
            * VALIDATION_FRACTION
        )
    ),
)

val_patients = set(
    patients[
        :num_val_patients
    ]
)

train_frame = frame[
    ~frame["patient_id"].isin(
        val_patients
    )
].reset_index(
    drop=True
)

val_frame = frame[
    frame["patient_id"].isin(
        val_patients
    )
].reset_index(
    drop=True
)

if (
    train_frame.empty
    or val_frame.empty
):
    raise RuntimeError(
        "Patient-level split created "
        "an empty dataset."
    )

print(
    "\nPatient-level split:"
)
print(
    "Total patients :",
    len(patients),
)
print(
    "Train patients :",
    train_frame[
        "patient_id"
    ].nunique(),
)
print(
    "Val patients   :",
    val_frame[
        "patient_id"
    ].nunique(),
)
print(
    "Train images   :",
    len(train_frame),
)
print(
    "Val images     :",
    len(val_frame),
)


# ============================================================
# 5. Dataset
# ============================================================

class UltrasoundSegmentationDataset(
    Dataset
):
    def __init__(
        self,
        data_frame,
        image_dir,
        mask_dir,
        training,
        output_size,
    ):
        self.frame = (
            data_frame
            .reset_index(drop=True)
        )

        self.image_dir = Path(
            image_dir
        )

        self.mask_dir = Path(
            mask_dir
        )

        self.training = bool(
            training
        )

        self.output_height = int(
            output_size[0]
        )

        self.output_width = int(
            output_size[1]
        )

    def __len__(self):
        return len(
            self.frame
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.frame.iloc[
            index
        ]

        image_path = (
            self.image_dir
            / str(row["image"])
        )

        mask_path = (
            self.mask_dir
            / str(row["mask"])
        )

        if not image_path.is_file():
            raise FileNotFoundError(
                image_path
            )

        if not mask_path.is_file():
            raise FileNotFoundError(
                mask_path
            )

        with Image.open(
            image_path
        ) as image:
            image = image.convert(
                "L"
            )

            image = image.resize(
                (
                    self.output_width,
                    self.output_height,
                ),
                resample=Image.BILINEAR,
            )

            image_array = (
                np.asarray(
                    image,
                    dtype=np.float32,
                )
                / 255.0
            )

        with Image.open(
            mask_path
        ) as mask:
            mask = mask.convert(
                "L"
            )

            mask = mask.resize(
                (
                    self.output_width,
                    self.output_height,
                ),
                resample=Image.NEAREST,
            )

            mask_array = (
                np.asarray(
                    mask,
                    dtype=np.uint8,
                )
                > 0
            ).astype(
                np.float32
            )

        image_tensor = (
            torch.from_numpy(
                image_array
            )
            .unsqueeze(0)
        )

        mask_tensor = (
            torch.from_numpy(
                mask_array
            )
            .unsqueeze(0)
        )

        if self.training:
            if random.random() < 0.5:
                image_tensor = torch.flip(
                    image_tensor,
                    dims=[2],
                )

                mask_tensor = torch.flip(
                    mask_tensor,
                    dims=[2],
                )

            if random.random() < 0.5:
                image_tensor = torch.flip(
                    image_tensor,
                    dims=[1],
                )

                mask_tensor = torch.flip(
                    mask_tensor,
                    dims=[1],
                )

            rotation_k = random.randint(
                0,
                3,
            )

            if rotation_k:
                image_tensor = torch.rot90(
                    image_tensor,
                    k=rotation_k,
                    dims=[1, 2],
                )

                mask_tensor = torch.rot90(
                    mask_tensor,
                    k=rotation_k,
                    dims=[1, 2],
                )

            if random.random() < 0.35:
                gain = random.uniform(
                    0.9,
                    1.1,
                )

                bias = random.uniform(
                    -0.04,
                    0.04,
                )

                image_tensor = (
                    image_tensor
                    * gain
                    + bias
                ).clamp(
                    0.0,
                    1.0,
                )

        return {
            "image": (
                image_tensor.float()
            ),
            "mask": (
                mask_tensor.float()
            ),
        }


train_dataset = (
    UltrasoundSegmentationDataset(
        train_frame,
        image_root,
        mask_root,
        training=True,
        output_size=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
    )
)

val_dataset = (
    UltrasoundSegmentationDataset(
        val_frame,
        image_root,
        mask_root,
        training=False,
        output_size=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
    )
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(
        device.type == "cuda"
    ),
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        device.type == "cuda"
    ),
    drop_last=False,
)


# ============================================================
# 6. U-Net implementation
# ============================================================

class DoubleConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
        )

    def forward(
        self,
        x,
    ):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(
                in_channels,
                out_channels,
            ),
        )

    def forward(
        self,
        x,
    ):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.conv = DoubleConv(
            out_channels
            + skip_channels,
            out_channels,
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = self.up(x)

        if x.shape[-2:] != (
            skip.shape[-2:]
        ):
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        x = torch.cat(
            [
                skip,
                x,
            ],
            dim=1,
        )

        return self.conv(x)


class RecommendedUNet(nn.Module):
    def __init__(
        self,
        input_channels=1,
        output_channels=1,
        base_channels=16,
    ):
        super().__init__()

        self.encoder1 = DoubleConv(
            input_channels,
            base_channels,
        )

        self.encoder2 = DownBlock(
            base_channels,
            base_channels * 2,
        )

        self.encoder3 = DownBlock(
            base_channels * 2,
            base_channels * 4,
        )

        self.bottleneck = DownBlock(
            base_channels * 4,
            base_channels * 8,
        )

        self.decoder3 = UpBlock(
            base_channels * 8,
            base_channels * 4,
            base_channels * 4,
        )

        self.decoder2 = UpBlock(
            base_channels * 4,
            base_channels * 2,
            base_channels * 2,
        )

        self.decoder1 = UpBlock(
            base_channels * 2,
            base_channels,
            base_channels,
        )

        self.output_head = nn.Conv2d(
            base_channels,
            output_channels,
            kernel_size=1,
        )

    def forward(
        self,
        x,
    ):
        skip1 = self.encoder1(x)
        skip2 = self.encoder2(skip1)
        skip3 = self.encoder3(skip2)

        bottleneck = self.bottleneck(
            skip3
        )

        x = self.decoder3(
            bottleneck,
            skip3,
        )

        x = self.decoder2(
            x,
            skip2,
        )

        x = self.decoder1(
            x,
            skip1,
        )

        return self.output_head(x)


model = RecommendedUNet(
    input_channels=1,
    output_channels=1,
    base_channels=BASE_CHANNELS,
).to(device)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "\nRecommended architecture:",
    RECOMMENDED_ARCHITECTURE,
)

print(
    "Notebook implementation:",
    "RecommendedUNet",
)

print(
    "Total parameters:",
    total_params,
)

print(
    "Trainable parameters:",
    trainable_params,
)

if total_params < 100_000:
    raise RuntimeError(
        "The segmentation model is "
        "unexpectedly small."
    )


# ============================================================
# 7. Segmentation loss and Dice metric
# ============================================================

bce_loss_function = (
    nn.BCEWithLogitsLoss()
)


def soft_dice_loss(
    logits,
    targets,
    epsilon=1e-6,
):
    probabilities = (
        torch.sigmoid(logits)
        .flatten(start_dim=1)
    )

    targets = targets.flatten(
        start_dim=1
    )

    intersection = (
        probabilities
        * targets
    ).sum(dim=1)

    denominator = (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
    )

    dice = (
        2.0 * intersection
        + epsilon
    ) / (
        denominator
        + epsilon
    )

    return (
        1.0
        - dice.mean()
    )


def segmentation_loss(
    logits,
    targets,
):
    loss_lower = (
        LOSS_NAME.lower()
    )

    if (
        "bce" in loss_lower
        and "dice" in loss_lower
    ):
        return (
            0.5
            * bce_loss_function(
                logits,
                targets,
            )
            + 0.5
            * soft_dice_loss(
                logits,
                targets,
            )
        )

    if "focal" in loss_lower:
        bce = (
            F.binary_cross_entropy_with_logits(
                logits,
                targets,
                reduction="none",
            )
        )

        probability = (
            torch.sigmoid(logits)
        )

        pt = (
            probability * targets
            + (1.0 - probability)
            * (1.0 - targets)
        )

        focal = (
            (
                (1.0 - pt) ** 2.0
            )
            * bce
        ).mean()

        return (
            0.5 * focal
            + 0.5
            * soft_dice_loss(
                logits,
                targets,
            )
        )

    if "dice" in loss_lower:
        return soft_dice_loss(
            logits,
            targets,
        )

    return (
        0.5
        * bce_loss_function(
            logits,
            targets,
        )
        + 0.5
        * soft_dice_loss(
            logits,
            targets,
        )
    )


def dice_scores(
    logits,
    targets,
    threshold,
    epsilon=1e-6,
):
    predictions = (
        torch.sigmoid(logits)
        >= threshold
    ).float()

    predictions = predictions.flatten(
        start_dim=1
    )

    targets = targets.flatten(
        start_dim=1
    )

    intersection = (
        predictions
        * targets
    ).sum(dim=1)

    denominator = (
        predictions.sum(dim=1)
        + targets.sum(dim=1)
    )

    scores = (
        2.0 * intersection
        + epsilon
    ) / (
        denominator
        + epsilon
    )

    both_empty = (
        denominator == 0
    )

    return torch.where(
        both_empty,
        torch.ones_like(scores),
        scores,
    )


# ============================================================
# 8. Optimizer and scheduler
# ============================================================

if OPTIMIZER_NAME == "adam":
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

elif OPTIMIZER_NAME == "sgd":
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,
    )

else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

scheduler = (
    torch.optim.lr_scheduler
    .CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-6,
    )
)

use_amp = (
    device.type == "cuda"
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)


# ============================================================
# 9. Resume from existing best checkpoint
# ============================================================

if not best_model_path.is_file():
    raise FileNotFoundError(
        "Existing best checkpoint was not found:\n"
        f"{best_model_path}\n"
        "Run the initial training Cell first."
    )

resume_checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False,
)

if "model_state_dict" not in resume_checkpoint:
    raise KeyError(
        "Checkpoint does not contain "
        "model_state_dict."
    )

model.load_state_dict(
    resume_checkpoint[
        "model_state_dict"
    ]
)

if (
    "optimizer_state_dict"
    in resume_checkpoint
):
    optimizer.load_state_dict(
        resume_checkpoint[
            "optimizer_state_dict"
        ]
    )

if (
    "scheduler_state_dict"
    in resume_checkpoint
):
    scheduler.load_state_dict(
        resume_checkpoint[
            "scheduler_state_dict"
        ]
    )

resumed_epoch = int(
    resume_checkpoint.get(
        "best_epoch",
        0,
    )
)

start_epoch = (
    resumed_epoch + 1
)

best_epoch = resumed_epoch

best_dice = float(
    resume_checkpoint.get(
        "best_metric",
        resume_checkpoint.get(
            "validation",
            {},
        ).get(
            "metric_value",
            -math.inf,
        ),
    )
)

if start_epoch > EPOCHS:
    raise RuntimeError(
        f"Checkpoint epoch {resumed_epoch} "
        f"has already reached the recommended "
        f"total of {EPOCHS} epochs."
    )

print(
    "\n=== RESUME TRAINING ==="
)

print(
    "Checkpoint:",
    best_model_path,
)

print(
    "Checkpoint epoch:",
    resumed_epoch,
)

print(
    "Existing best Dice:",
    best_dice,
)

print(
    "Continue from epoch:",
    start_epoch,
)

print(
    "Final recommended epoch:",
    EPOCHS,
)

print(
    "Early stopping:",
    "disabled for this continuation run",
)


# ============================================================
# 10. Validation
# ============================================================

def evaluate_model():
    model.eval()

    total_loss = 0.0
    total_samples = 0
    scores = []

    with torch.inference_mode():
        for batch in val_loader:
            images = batch[
                "image"
            ].to(
                device,
                non_blocking=True,
            )

            masks = batch[
                "mask"
            ].to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(
                    images
                )

                loss = (
                    segmentation_loss(
                        logits,
                        masks,
                    )
                )

            batch_size = (
                images.shape[0]
            )

            total_loss += (
                float(loss.item())
                * batch_size
            )

            total_samples += (
                batch_size
            )

            scores.extend(
                dice_scores(
                    logits.float(),
                    masks,
                    PREDICTION_THRESHOLD,
                )
                .cpu()
                .tolist()
            )

    return {
        "loss": (
            total_loss
            / max(
                total_samples,
                1,
            )
        ),
        "dice": float(
            np.mean(scores)
        ),
        "num_samples": (
            total_samples
        ),
    }


# ============================================================
# 11. Continue training until EPOCHS
# ============================================================

epochs_without_improvement = 0
training_start = time.time()

print(
    f"\nContinuing "
    f"{RECOMMENDED_ARCHITECTURE} "
    f"from epoch {start_epoch} "
    f"through epoch {EPOCHS}..."
)

with training_log_path.open(
    "a",
    encoding="utf-8",
    buffering=1,
) as log_file:

    log_file.write(
        "\n===== CONTINUATION RUN START "
        f"{datetime.now().isoformat()} "
        "=====\n"
    )

    log_file.write(
        f"Resumed from epoch {resumed_epoch}; "
        f"existing best Dice={best_dice}\n"
    )

    for epoch in range(
        start_epoch,
        EPOCHS + 1,
    ):
        epoch_start = time.time()

        model.train()

        running_loss = 0.0
        seen_samples = 0

        for batch in train_loader:
            images = batch[
                "image"
            ].to(
                device,
                non_blocking=True,
            )

            masks = batch[
                "mask"
            ].to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(
                    images
                )

                if (
                    logits.shape
                    != masks.shape
                ):
                    raise RuntimeError(
                        "Segmentation output and mask "
                        "shapes do not match:\n"
                        f"output={tuple(logits.shape)}\n"
                        f"mask={tuple(masks.shape)}"
                    )

                loss = segmentation_loss(
                    logits,
                    masks,
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            batch_size = (
                images.shape[0]
            )

            running_loss += (
                float(loss.item())
                * batch_size
            )

            seen_samples += (
                batch_size
            )

        scheduler.step()

        train_loss = (
            running_loss
            / max(
                seen_samples,
                1,
            )
        )

        validation = (
            evaluate_model()
        )

        val_loss = float(
            validation["loss"]
        )

        val_dice = float(
            validation["dice"]
        )

        current_lr = float(
            optimizer.param_groups[
                0
            ]["lr"]
        )

        epoch_time = (
            time.time()
            - epoch_start
        )

        line = (
            f"[train] epoch {epoch}/{EPOCHS}  "
            f"loss={train_loss:.6f}  "
            f"val_loss={val_loss:.6f}  "
            f"val_dice={val_dice:.6f}  "
            f"lr={current_lr:.2e}  "
            f"steps={len(train_loader)}  "
            f"time={epoch_time:.1f}s"
        )

        print(
            line,
            flush=True,
        )

        log_file.write(
            line + "\n"
        )

        record = {
            "epoch": epoch,
            "total_epochs": EPOCHS,
            "train_loss": (
                train_loss
            ),
            "val_loss": (
                val_loss
            ),
            "val_dice": (
                val_dice
            ),
            "learning_rate": (
                current_lr
            ),
            "recommended_architecture": (
                RECOMMENDED_ARCHITECTURE
            ),
            "recommended_loss": (
                LOSS_NAME
            ),
            "recommended_optimizer": (
                OPTIMIZER_NAME
            ),
            "continuation_run": True,
            "resumed_from_epoch": (
                resumed_epoch
            ),
            "timestamp": (
                datetime.now()
                .isoformat()
            ),
        }

        with metrics_path.open(
            "a",
            encoding="utf-8",
        ) as metrics_file:
            metrics_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch
            epochs_without_improvement = 0

            checkpoint = {
                "model_state_dict": (
                    model.state_dict()
                ),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),
                "best_epoch": (
                    best_epoch
                ),
                "best_metric": (
                    best_dice
                ),
                "validation": {
                    "metric_name": (
                        "dice"
                    ),
                    "metric_value": (
                        best_dice
                    ),
                    "dice": (
                        best_dice
                    ),
                    "loss": (
                        val_loss
                    ),
                    "num_samples": int(
                        validation[
                            "num_samples"
                        ]
                    ),
                    "empty_mask_policy": (
                        "both empty equals Dice 1"
                    ),
                },
                "config": {
                    **recommended_cfg,
                    "recommended_architecture": (
                        RECOMMENDED_ARCHITECTURE
                    ),
                    "implemented_model_class": (
                        "RecommendedUNet"
                    ),
                    "recommended_epochs": (
                        EPOCHS
                    ),
                    "batch_size": (
                        BATCH_SIZE
                    ),
                    "learning_rate": (
                        LEARNING_RATE
                    ),
                    "weight_decay": (
                        WEIGHT_DECAY
                    ),
                    "image_height": (
                        IMAGE_HEIGHT
                    ),
                    "image_width": (
                        IMAGE_WIDTH
                    ),
                    "base_channels": (
                        BASE_CHANNELS
                    ),
                    "prediction_threshold": (
                        PREDICTION_THRESHOLD
                    ),
                    "loss": (
                        LOSS_NAME
                    ),
                    "optimizer": (
                        OPTIMIZER_NAME
                    ),
                    "validation_patients": (
                        sorted(
                            val_patients
                        )
                    ),
                },
                "model_class": (
                    "RecommendedUNet"
                ),
                "task_type": (
                    "image_segmentation"
                ),
                "total_params": (
                    total_params
                ),
                "trainable_params": (
                    trainable_params
                ),
            }

            torch.save(
                checkpoint,
                best_model_path,
            )

            representative_path = (
                important_dir
                / (
                    "best_epoch_"
                    f"{epoch:03d}.pt"
                )
            )

            shutil.copy2(
                best_model_path,
                representative_path,
            )

            print(
                "[train] Saved new best "
                f"checkpoint at epoch {epoch}: "
                f"{best_model_path}",
                flush=True,
            )

            print(
                "[pipeline] Preserved "
                "representative checkpoint:",
                representative_path,
                flush=True,
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            message = (
                "[train] No validation improvement "
                f"for {EARLY_STOPPING_PATIENCE} epochs, "
                f"but training will continue until "
                f"epoch {EPOCHS}."
            )

            print(
                message,
                flush=True,
            )

            log_file.write(
                message + "\n"
            )

    log_file.write(
        "===== CONTINUATION RUN END "
        f"{datetime.now().isoformat()} "
        "=====\n"
    )


# ============================================================
# 12. Final result
# ============================================================

if not best_model_path.is_file():
    raise RuntimeError(
        "best_model.pt is missing "
        "after continuation training."
    )

print(
    "\nContinuation training finished."
)

print(
    "RAG-recommended architecture:",
    RECOMMENDED_ARCHITECTURE,
)

print(
    "Recommended total epochs:",
    EPOCHS,
)

print(
    "Resumed from epoch:",
    resumed_epoch,
)

print(
    "Final completed epoch:",
    EPOCHS,
)

print(
    "Best checkpoint:",
    best_model_path,
)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation Dice:",
    best_dice,
)

print(
    "Representative checkpoints:",
    important_dir,
)

print(
    "Epoch metrics:",
    metrics_path,
)

print(
    "Training log:",
    training_log_path,
)

print(
    "Continuation runtime:",
    round(
        time.time()
        - training_start,
        1,
    ),
    "seconds",
)


=== PIPELINE RECOMMENDATION USED FOR TRAINING ===
architecture            : unet
recommended_epochs      : 30
batch_size              : 8
learning_rate           : 0.001
weight_decay            : 1e-05
image_size              : (224, 224)
optimizer               : adam
loss                    : bce_dice_loss
early_stopping_patience : 8
prediction_threshold    : 0.5
base_channels           : 16
validation_fraction     : 0.2

Device -> cuda
Manifest -> /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_ultrasound_segmentation.csv
Images -> /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_images
Masks -> /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/jiaozi_train_masks
Checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/ultrasound_nerve_dice
Representative checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/ultrasound_nerve_dice/important_epochs
Resume checkpoint -> enabled
evaluation_metric -> dice

Patient-level split:
To

In [ ]:
!nvidia-smi


Sat Jul 11 22:46:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Show the result

Reads the best checkpoint and prints its validation score **instantly** (the metric was
already computed during training — no re-evaluation). Set `FULL_EVAL = True` for a full
re-evaluation (macro-F1, params, sample count), which costs one extra pass over the eval set.


In [20]:
# Show the best Ultrasound Nerve Segmentation result

import json
from pathlib import Path

import torch

best_path = Path(
    "/content/drive/MyDrive/"
    "Jiaozi/checkpoints/"
    "ultrasound_nerve_dice/"
    "best_model.pt"
)

if not best_path.is_file():
    raise FileNotFoundError(
        f"best_model.pt not found: {best_path}"
    )

checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False,
)

validation = (
    checkpoint.get("validation")
    or {}
)

metric_value = (
    validation.get("metric_value")
)

if metric_value is None:
    metric_value = checkpoint.get(
        "best_metric"
    )

metric_name = str(
    validation.get(
        "metric_name",
        "dice",
    )
).lower()

best_epoch = checkpoint.get(
    "best_epoch"
)

print(
    "=== RESULT "
    "(best Ultrasound checkpoint) ==="
)

print("checkpoint :", best_path)
print("best_epoch :", best_epoch)
print(
    "best_metric:",
    metric_value,
    "(highest validation Dice)",
)

print(
    "validation :",
    json.dumps(
        validation,
        indent=2,
        ensure_ascii=False,
    ),
)

if metric_name not in {
    "dice",
    "dice_score",
    "dice_coefficient",
}:
    raise RuntimeError(
        "Expected Dice metric, "
        f"but checkpoint reports {metric_name!r}."
    )

if metric_value is None:
    raise RuntimeError(
        "No validation Dice value was found."
    )

metric_value = float(metric_value)

if not 0.0 <= metric_value <= 1.0:
    raise RuntimeError(
        f"Invalid Dice value: {metric_value}"
    )

print(
    "\nBest validation Dice:",
    metric_value,
)

print(
    "Higher Dice is better."
)


=== RESULT (best Ultrasound checkpoint) ===
checkpoint : /content/drive/MyDrive/Jiaozi/checkpoints/ultrasound_nerve_dice/best_model.pt
best_epoch : 14
best_metric: 0.40356299970997583 (highest validation Dice)
validation : {
  "metric_name": "dice",
  "metric_value": 0.40356299970997583,
  "dice": 0.40356299970997583,
  "loss": 0.3957365016566039,
  "num_samples": 1079,
  "empty_mask_policy": "both empty equals Dice 1"
}

Best validation Dice: 0.40356299970997583
Higher Dice is better.


In [22]:
# Generate Kaggle Ultrasound Nerve Segmentation submission.csv
# Uses the RecommendedUNet checkpoint produced by the training Cell.
# This Cell does not retrain the model.

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image


# ============================================================
# 1. Paths
# ============================================================

OUTPUT_DIR = Path(
    "/content/jiaozi_generated_pipeline"
)

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/Jiaozi/checkpoints/"
    "ultrasound_nerve_dice/best_model.pt"
)

SUBMISSION_PATH = (
    OUTPUT_DIR / "submission.csv"
)

DATA_ROOT = Path(
    "/content/data/ultrasound-nerve/"
    "ultrasound-nerve-segmentation"
)

if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"Generated project not found: {OUTPUT_DIR}"
    )

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}"
    )

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Competition data not found: {DATA_ROOT}"
    )


# ============================================================
# 2. Locate sample submission and test images
# ============================================================

sample_candidates = sorted(
    path
    for path in DATA_ROOT.rglob("*.csv")
    if "sample" in path.name.lower()
    and "submission" in path.name.lower()
)

if not sample_candidates:
    raise FileNotFoundError(
        "Could not locate sample_submission.csv."
    )

sample_path = sample_candidates[0]

test_dir_candidates = sorted(
    path
    for path in DATA_ROOT.rglob("*")
    if path.is_dir()
    and path.name.lower() == "test"
)

TEST_DIR = None
test_images = []

IMAGE_EXTENSIONS = {
    ".tif",
    ".tiff",
    ".png",
    ".jpg",
    ".jpeg",
}

for candidate in test_dir_candidates:
    candidate_images = sorted(
        path
        for path in candidate.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
        and not path.stem.endswith("_mask")
    )

    if candidate_images:
        TEST_DIR = candidate
        test_images = candidate_images
        break

if TEST_DIR is None:
    raise FileNotFoundError(
        f"Could not locate test images under {DATA_ROOT}"
    )

sample = pd.read_csv(
    sample_path
)

if sample.shape[1] != 2:
    raise ValueError(
        "Expected a two-column sample submission, "
        f"received shape {sample.shape}."
    )

id_column = sample.columns[0]
pixels_column = sample.columns[1]

print("Sample submission :", sample_path)
print("Sample shape      :", sample.shape)
print("Columns           :", sample.columns.tolist())
print("Test directory    :", TEST_DIR)
print("Test images       :", len(test_images))
print("Checkpoint        :", CHECKPOINT_PATH)


# ============================================================
# 3. Load checkpoint and training configuration
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

checkpoint_config = checkpoint.get(
    "config",
    {},
)

if not isinstance(
    checkpoint_config,
    dict,
):
    checkpoint_config = {}

BASE_CHANNELS = int(
    checkpoint_config.get(
        "base_channels",
        16,
    )
)

IMAGE_HEIGHT = int(
    checkpoint_config.get(
        "image_height",
        256,
    )
)

IMAGE_WIDTH = int(
    checkpoint_config.get(
        "image_width",
        256,
    )
)

PREDICTION_THRESHOLD = float(
    checkpoint_config.get(
        "prediction_threshold",
        0.5,
    )
)

print("\nDevice               :", device)
print("Model class          :", checkpoint.get("model_class"))
print("Best epoch           :", checkpoint.get("best_epoch"))
print("Best validation Dice :", checkpoint.get("best_metric"))
print("Base channels        :", BASE_CHANNELS)
print(
    "Inference size       :",
    (IMAGE_HEIGHT, IMAGE_WIDTH),
)
print(
    "Prediction threshold :",
    PREDICTION_THRESHOLD,
)


# ============================================================
# 4. Same U-Net architecture used during training
# ============================================================

class DoubleConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.ReLU(
                inplace=True
            ),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(
                in_channels,
                out_channels,
            ),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels,
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = self.up(x)

        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        x = torch.cat(
            [skip, x],
            dim=1,
        )

        return self.conv(x)


class RecommendedUNet(nn.Module):
    def __init__(
        self,
        input_channels=1,
        output_channels=1,
        base_channels=16,
    ):
        super().__init__()

        self.encoder1 = DoubleConv(
            input_channels,
            base_channels,
        )

        self.encoder2 = DownBlock(
            base_channels,
            base_channels * 2,
        )

        self.encoder3 = DownBlock(
            base_channels * 2,
            base_channels * 4,
        )

        self.bottleneck = DownBlock(
            base_channels * 4,
            base_channels * 8,
        )

        self.decoder3 = UpBlock(
            base_channels * 8,
            base_channels * 4,
            base_channels * 4,
        )

        self.decoder2 = UpBlock(
            base_channels * 4,
            base_channels * 2,
            base_channels * 2,
        )

        self.decoder1 = UpBlock(
            base_channels * 2,
            base_channels,
            base_channels,
        )

        self.output_head = nn.Conv2d(
            base_channels,
            output_channels,
            kernel_size=1,
        )

    def forward(self, x):
        skip1 = self.encoder1(x)
        skip2 = self.encoder2(skip1)
        skip3 = self.encoder3(skip2)

        bottleneck = self.bottleneck(
            skip3
        )

        x = self.decoder3(
            bottleneck,
            skip3,
        )

        x = self.decoder2(
            x,
            skip2,
        )

        x = self.decoder1(
            x,
            skip1,
        )

        return self.output_head(x)


# ============================================================
# 5. Build model and load checkpoint
# ============================================================

model = RecommendedUNet(
    input_channels=1,
    output_channels=1,
    base_channels=BASE_CHANNELS,
).to(device)

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "Checkpoint does not contain model_state_dict."
    )

state_dict = checkpoint[
    "model_state_dict"
]

missing_keys, unexpected_keys = (
    model.load_state_dict(
        state_dict,
        strict=False,
    )
)

print("\nMissing keys   :", missing_keys)
print("Unexpected keys:", unexpected_keys)

if missing_keys or unexpected_keys:
    raise RuntimeError(
        "Checkpoint is incompatible with RecommendedUNet.\n"
        f"Missing keys: {missing_keys}\n"
        f"Unexpected keys: {unexpected_keys}"
    )

model.eval()

print(
    "Checkpoint loaded successfully."
)


# ============================================================
# 6. Prediction
# ============================================================

def predict_mask(
    image_path: Path,
):
    with Image.open(
        image_path
    ) as image:
        grayscale = image.convert(
            "L"
        )

        original_width, original_height = (
            grayscale.size
        )

        resized = grayscale.resize(
            (
                IMAGE_WIDTH,
                IMAGE_HEIGHT,
            ),
            resample=Image.BILINEAR,
        )

        image_array = (
            np.asarray(
                resized,
                dtype=np.float32,
            )
            / 255.0
        )

    input_tensor = (
        torch.from_numpy(
            image_array
        )
        .unsqueeze(0)
        .unsqueeze(0)
        .to(device)
    )

    with torch.inference_mode():
        logits = model(
            input_tensor
        )

        probabilities = torch.sigmoid(
            logits.float()
        )

        probabilities = F.interpolate(
            probabilities,
            size=(
                original_height,
                original_width,
            ),
            mode="bilinear",
            align_corners=False,
        )

        binary_mask = (
            probabilities[0, 0]
            >= PREDICTION_THRESHOLD
        )

    return (
        binary_mask
        .cpu()
        .numpy()
        .astype(np.uint8)
    )


# ============================================================
# 7. Kaggle RLE encoding
# ============================================================

def rle_encode(
    mask: np.ndarray,
):
    # Kaggle expects column-major order.
    pixels = mask.flatten(
        order="F"
    )

    pixels = np.concatenate(
        [
            np.array(
                [0],
                dtype=np.uint8,
            ),
            pixels,
            np.array(
                [0],
                dtype=np.uint8,
            ),
        ]
    )

    changes = np.where(
        pixels[1:]
        != pixels[:-1]
    )[0] + 1

    starts = changes[::2]
    ends = changes[1::2]

    lengths = (
        ends - starts
    )

    if len(starts) == 0:
        return ""

    return " ".join(
        str(value)
        for pair in zip(
            starts,
            lengths,
        )
        for value in pair
    )


# ============================================================
# 8. Match sample IDs to test images
# ============================================================

test_image_map = {
    path.stem: path
    for path in test_images
}


def resolve_test_image(
    raw_id,
):
    image_id = Path(
        str(raw_id)
    ).stem

    if image_id in test_image_map:
        return test_image_map[
            image_id
        ]

    # Handle IDs such as 1 versus 0001.
    stripped_id = (
        image_id.lstrip("0")
        or "0"
    )

    for candidate_id, candidate_path in (
        test_image_map.items()
    ):
        candidate_stripped = (
            candidate_id.lstrip("0")
            or "0"
        )

        if candidate_stripped == stripped_id:
            return candidate_path

    return None


encoded_masks = []

for index, raw_id in enumerate(
    sample[id_column],
    start=1,
):
    image_path = resolve_test_image(
        raw_id
    )

    if image_path is None:
        raise FileNotFoundError(
            "Could not find the test image "
            f"for sample ID {raw_id!r}."
        )

    predicted_mask = predict_mask(
        image_path
    )

    encoded_masks.append(
        rle_encode(
            predicted_mask
        )
    )

    if (
        index % 500 == 0
        or index == len(sample)
    ):
        print(
            f"Predicted {index}/{len(sample)} images",
            flush=True,
        )


# ============================================================
# 9. Build and validate submission
# ============================================================

submission = sample.copy()

submission[
    pixels_column
] = encoded_masks

if submission.shape != sample.shape:
    raise RuntimeError(
        "Submission shape does not match "
        "the official sample submission."
    )

if list(
    submission.columns
) != list(
    sample.columns
):
    raise RuntimeError(
        "Submission columns do not match "
        "the official sample submission."
    )

if (
    submission[id_column]
    .astype(str)
    .tolist()
    != sample[id_column]
    .astype(str)
    .tolist()
):
    raise RuntimeError(
        "Submission ID order does not match "
        "the official sample submission."
    )

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
)

non_empty = int(
    submission[
        pixels_column
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.len()
    .gt(0)
    .sum()
)

print(
    "\n✅ Ultrasound Nerve Segmentation "
    "submission generated successfully."
)

print(
    "Submission path :",
    SUBMISSION_PATH,
)

print(
    "Submission shape:",
    submission.shape,
)

print(
    "Non-empty masks :",
    non_empty,
    "/",
    len(submission),
)

display(
    submission.head()
)


Sample submission : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/sample_submission.csv
Sample shape      : (5508, 2)
Columns           : ['img', 'pixels']
Test directory    : /content/data/ultrasound-nerve/ultrasound-nerve-segmentation/test
Test images       : 5508
Checkpoint        : /content/drive/MyDrive/Jiaozi/checkpoints/ultrasound_nerve_dice/best_model.pt

Device               : cuda
Model class          : RecommendedUNet
Best epoch           : 14
Best validation Dice : 0.40356299970997583
Base channels        : 16
Inference size       : (224, 224)
Prediction threshold : 0.5

Missing keys   : []
Unexpected keys: []
Checkpoint loaded successfully.
Predicted 500/5508 images
Predicted 1000/5508 images
Predicted 1500/5508 images
Predicted 2000/5508 images
Predicted 2500/5508 images
Predicted 3000/5508 images
Predicted 3500/5508 images
Predicted 4000/5508 images
Predicted 4500/5508 images
Predicted 5000/5508 images
Predicted 5500/5508 images
Predicted 5508/5508 images


,img,pixels
0,1,114348 1 114768 1 115187 2 115607 2 116028 1 1...
1,2,90426 4 90844 7 91263 9 91682 11 92102 11 9252...
2,3,128182 1 128186 4 128601 10 129021 11 129441 1...
3,4,41759 6 42178 9 42597 11 42633 1 43015 14 4305...
4,5,102219 11 102629 28 103048 30 103467 34 103887...


In [23]:
# Submit to Kaggle under Ultrasound Nerve Segmentation

!kaggle competitions submit \
    -c ultrasound-nerve-segmentation \
    -f /content/jiaozi_generated_pipeline/submission.csv \
    -m "Jiaozi Ultrasound Nerve Segmentation Dice submission"


100% 3.32M/3.32M [00:02<00:00, 1.55MB/s]
Successfully submitted to Ultrasound Nerve Segmentation

In [24]:
!kaggle competitions submissions \
    -c ultrasound-nerve-segmentation


     ref  fileName        date                        description                                           status                     publicScore  privateScore  
--------  --------------  --------------------------  ----------------------------------------------------  -------------------------  -----------  ------------  
54616083  submission.csv  2026-07-12 16:54:07.270000  Jiaozi Ultrasound Nerve Segmentation Dice submission  SubmissionStatus.COMPLETE  0.47818      0.45699       
